# FLARE: Fire Lineage and Reconstruction Engine  
## Pantanal Wildfire Event Reconstruction Workflow

**Author**: Tianjiao Pu (TJ)  
**Current Affiliation**: High Meadows Environmental Institute, Princeton University  
**Project**: A Song of Water and Fire — Berkeley FLARE  

**Last updated**: March 2026  

---

This notebook demonstrates how to reconstruct wildfire events in the Pantanal 
using the **FLARE (Fire Lineage and Reconstruction Engine)** framework.

FLARE converts daily burned area maps into a structured fire event database by:

1. Clustering burned pixels into daily fire objects
2. Tracking fire objects across time
3. Reconstructing fire lineages and fire events
4. Storing event information in a queryable SQLite database

The resulting database enables rapid analysis of wildfire dynamics, including:
- identification of the largest fire events
- reconstruction of fire trajectories
- analysis of ignition patterns

## Workflow Overview

The reconstruction pipeline consists of four main steps:

| Step | Description | Output |
|-----|-------------|------|
| 1 | Fire object clustering | Daily fire objects |
| 2 | Temporal object linking | Parent–child object relationships |
| 3 | Link validation | Verified object lineage graph |
| 4 | Database construction | Fire event SQLite database |

The final database organizes wildfire activity into **event-based structures** 
that allow efficient retrieval of fire events and their full temporal evolution.

In [ ]:
# --- Environment setup
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent))

# --- Standard libraries
import importlib
import calendar
import sqlite3
import pandas as pd
import scipy.sparse as sp

# --- Import FLARE(Fire Lineage and Reconstruction Engine) core module
import flare_event_reconstruction
from flare_event_reconstruction import run_dbscan_year,track_objects, build_year_fire_db

## Step 1 — Fire Object Clustering
 
Identify spatially connected burned pixels and group them into **daily fire objects**.

Inputs
- Pseudo-Daily burned area dataset (`Burned_Area_Daily_YearYYYY.npz`)

Outputs
- Clustered daily fire objects (`Burned_Area_Daily_DBSCAN_YearYYYY.npz`)

In [ ]:
# --- 1. User Configufration ---
year = 2001
npy_shape = (1792, 1671)  # Pantanal domain resolution for lat lon extent [-61.5, -54, -22, -14]

# --- 2. Path Management ---
input_file = f"input/Burned_Area_Daily_Year{year}.npz"
dbscan_file = f"intermediate/Burned_Area_Daily_DBSCAN_Year{year}.npz"

# --- 3. Auto-Params ---
days_in_year = 366 if calendar.isleap(year) else 365

print(f"Initializing Step 1 for Year {year}")
print(f"   - Target: {days_in_year} days tracking")
print(f"   - Domain Shape: {npy_shape}")

# --- 4. Core Execution ---
print(f"Clustering fire objects (DBSCAN)... This might take a while.")
run_dbscan_year(input_file, dbscan_file, npy_shape=npy_shape, eps=3, min_samples=3)
print(f"✅ Step 1 Complete! Intermediate file saved at: {dbscan_file}")

## Step 2 — Object Linking

Connect fire objects across consecutive days to reconstruct **fire trajectories**.

This process produces a **temporal graph of fire objects**, which represents 
the evolving structure of wildfire activity.

Inputs
- Clustered fire objects from Step 1

Outputs
- Object lineage links (`yearXXXX_object_link.npy`)

In [ ]:
# --- 1. Path Management ---
dbscan_file = f"intermediate/Burned_Area_Daily_DBSCAN_Year{year}.npz"
link_save_file = f"intermediate/year{year}_object_link.npy"

# --- 2. Loading ---
print(f"Step 2: Linking fire objects across {days_in_year} days...")
print(f"   - Loading intermediate cluster data: {dbscan_file}")
loaded_sparse_matrix = sp.load_npz(dbscan_file)
fire_dbscan_full = loaded_sparse_matrix.toarray().reshape((days_in_year, *npy_shape))

# --- 3. Object Tracking Loop ---
time_track = []
data_track = []
print(f"Tracking progress (this may take a few minutes)...")
for t1 in range(fire_dbscan_full.shape[0] - 1):
    frames = fire_dbscan_full[t1:t1+2]
    tracks = track_objects(frames)
    if t1 % 25 == 0 or t1 == (days_in_year - 2):
        print(f"   [Day {t1:03d}] Identified {len(tracks):4d} connected object links.")
    time_track.append(t1)
    data_track.append(tracks)

# --- 4. Persistence ---
converted_data = np.array([list(sublist) for sublist in data_track], dtype=object)
np.save(link_save_file, [time_track, converted_data])

print(f"\n Step 2 Complete!")
print(f"   - Relationship graph saved at: {link_save_file}")
print(f"   - Total link days processed: {len(time_track)}")

## Step 3 — Link Validation

Ensure that the reconstructed object links are consistent with the reference for this demo run.

In [ ]:
# --- 1. Path Management ---
current_link_file = f"intermediate/year{year}_object_link.npy"
validation_source = f"validation/valid_year{year}_object_link.npy"
print(f"Step 3: Validating Link Consistency...")

# Linking has structure [time_track, converted_data]
_, current_converted = np.load(current_link_file, allow_pickle=True)
valid_data_raw = np.load(validation_source, allow_pickle=True)
valid_converted = valid_data_raw[1]
print(f" - Comparing against: {validation_source}")

# --- 2. Compare ---
is_equal = np.array_equal(current_converted, valid_converted)

if is_equal:
    print("\n" + "="*30)
    print("VALIDATION PASSED!")
    print("   The generated links match the historical record 100%.")
    print("="*30)
else:
    print("\n" + "!"*30)
    print("VALIDATION FAILED!")
    print("   The current links differ from the validation file.")
    print("   Check if tracking parameters (eps, min_samples) have changed.")
    print("!"*30)

## Step 4 — Fire Event Database Construction

Convert the object lineage graph into a structured **fire event database**.
The database stores:

- fire objects
- object transitions
- fire events
- event temporal summaries
- spatial metadata

Output
- SQLite database (`pantanal_fire_events_YEAR.db`)

In [ ]:
fire_event_db = f"output/pantanal_fire_events_{year}.db"
build_year_fire_db(
    year=year,
    npz_path=dbscan_file,
    npy_path=link_save_file,
    db_path=fire_event_db,
    raster_shape=npy_shape,
)

## Quick Dataset Inspection

After the database is created, we perform a quick inspection to confirm that the
data structure was generated correctly.

We examine:
- project metadata
- database tables

In [ ]:
# --- 1. user configufration/ path ---
conn = sqlite3.connect(fire_event_db)

# --- 2.  read Metadata ---
print("--- Project Metadata ---")
metadata = pd.read_sql_query("SELECT * FROM project_metadata", conn)
print(metadata.to_string(index=False, justify='middle'))

# --- 3. check data structure ---
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("\n--- Tables in Database ---")
print(tables)